In [ ]:
!pip install unsloth -q
import os
if "COLAB_" in "".join(os.environ.keys()):
    !pip install --no-deps bitsandbytes accelerate peft trl triton cut_cross_entropy unsloth_zoo -q
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer -q
    !pip install --no-deps unsloth -q

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "Qwen/Qwen3-VL-2B-Instruct",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset

# HF dataset columns: image (PIL), system, question, answer, episode_id, step, total_steps, category
dataset = load_dataset("lakminaG/hm3d-objectnav-vln-sft-long-horizon", split="train")

print(f"Dataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"Sample system : {dataset[0]['system']}")
print(f"Sample question: {dataset[0]['question']}")
print(f"Sample answer  : {dataset[0]['answer']}")

In [ ]:
def format_data(sample):
    """
    Converts a flat HF dataset row into the conversation format Unsloth expects.

    HF dataset columns used:
      - sample["image"]    : PIL.Image  (stored directly in the dataset)
      - sample["system"]   : str        (system prompt)
      - sample["question"] : str        (user instruction with target category)
      - sample["answer"]   : str        (one of MOVE_FORWARD / TURN_LEFT / TURN_RIGHT / STOP)

    Note: Qwen3-VL chat template raises a TemplateError when a system message is
    combined with images. The system prompt is prepended to the user text instead.
    """
    combined_text = sample["system"] + "\n" + sample["question"]

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text",  "text": combined_text}
            ]
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["answer"]}]
        }
    ]

    return {"messages": conversation}

converted_dataset = [format_data(sample) for sample in dataset]
print(f"Converted {len(converted_dataset)} samples.")

In [ ]:
!pip install wandb -q

import wandb
from google.colab import userdata

wandb.login(key=userdata.get("WANDB_API_KEY"))

wandb.init(project="hm3d-objectnav-qwen3vl-2b")

In [ ]:
from unsloth import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 60,        # uncomment for a quick smoke-test
        num_train_epochs = 10,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        logging_strategy = "steps",
        report_to = "wandb",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

trainer_stats = trainer.train()

In [ ]:
# ── Inference on sample 0 ────────────────────────────────────────────────────
from transformers import TextStreamer

FastVisionModel.for_inference(model)

sample = dataset[0]
combined_text = sample["system"] + "\n" + sample["question"]

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": combined_text}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    sample["image"],
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print(f"Ground truth : {sample['answer']}")
print("Model prediction:")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 16,
    use_cache = True,
    temperature = 1.0,
    do_sample = False,
)

In [ ]:
# ── Inference on sample 1 ────────────────────────────────────────────────────
sample = dataset[1]
combined_text = sample["system"] + "\n" + sample["question"]

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": combined_text}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    sample["image"],
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

print(f"Ground truth : {sample['answer']}")
print("Model prediction:")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 16,
    use_cache = True,
    temperature = 1.0,
    do_sample = False,
)

In [ ]:
# ── Save & push to Hugging Face Hub ─────────────────────────────────────────
from huggingface_hub import login
from google.colab import userdata

repo_id = "lakminaG/hm3d-objectnav-Qwen3VL-2B-it-fine-tuned-v1"

print(f"\nPushing to Hugging Face Hub: {repo_id}...")

login(token=userdata.get("HF_TOKEN"))

# model.push_to_hub(repo_id)
# tokenizer.push_to_hub(repo_id)

# Upload merged 16-bit model weights
model.push_to_hub_merged(repo_id, tokenizer, save_method="merged_16bit")

print("Upload complete!")